# Practical 7: Vector Search — FAISS to OpenSearch


In [1]:
# sentence-transformers -> SBERT embeddings, shared by both sub-projects
# faiss-cpu               -> Part 1: local vector index
# boto3 / opensearch-py    -> Part 2: AWS OpenSearch Serverless
# !pip install -q sentence-transformers faiss-cpu boto3 opensearch-py python-dotenv

In [1]:
import sys, os

def use_subproject(subproject_path: str):
    """Switches the active 'src'/'utils' import namespace to a given
    sub-project folder -- see this notebook's intro cell for why this is
    needed."""
    for mod_name in list(sys.modules):
        if mod_name == "src" or mod_name.startswith("src.") \
           or mod_name == "utils" or mod_name.startswith("utils."):
            del sys.modules[mod_name]

    global _active_subproject_path
    if "_active_subproject_path" in globals() and _active_subproject_path in sys.path:
        sys.path.remove(_active_subproject_path)

    abs_path = os.path.abspath(subproject_path)
    sys.path.insert(0, abs_path)
    _active_subproject_path = abs_path
    print(f"Active sub-project: {abs_path}")

## Part 1 — Local vector search with FAISS

### The dataset
100 sentences across 5 topics (20 each): Football, Technology,
cricket, Travel & Places, Health & Wellness — see
`faiss_search/src/data.py`'s docstring for why this shape.

In [2]:
use_subproject("../faiss_search")

from src.data import get_sentences, get_topics
from src.embedder import encode_sentences
from src.faiss_index import build_faiss_index, search

sentences = get_sentences()
topics = get_topics()
print(f"{len(sentences)} sentences across {len(set(topics))} topics")

Active sub-project: /Users/pateljay/Documents/GenAI/practical_07_vector_search_faiss_opensearch/faiss_search
100 sentences across 5 topics


In [3]:
import requests
print(requests.get("https://huggingface.co").status_code)

200


### Encode all 100 sentences and build the index

In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = encode_sentences(sentences)
print("Embeddings shape:", embeddings.shape)

index = build_faiss_index(embeddings)
print("FAISS index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (100, 384)
FAISS index size: 100


### Run similarity search for a few queries

In [5]:
queries = [
    "Cristiano Ronaldo, a towering monument of athletic perfection.",
    "Vaibhav is magnificient at hitting boundaries at this age.",
    "Engineers deployed the new app update to all users.",
    "She stretched before her morning run.",
]

for query in queries:
    query_embedding = encode_sentences([query])[0]
    results = search(index, query_embedding, sentences, k=3)
    print(f"QUERY: {query}")
    for rank, (sentence, score) in enumerate(results, start=1):
        print(f"  {rank}. ({score:.3f}) {sentence}")
    print()

QUERY: Cristiano Ronaldo, a towering monument of athletic perfection.
  1. (0.819) Cristiano Ronaldo, a towering monument of athletic perfection, leaps into the heavens to plant a thunderous header into the net!
  2. (0.484) Lionel Messi, the diminutive magician of Rosario, weaves through a forest of legs to paint his masterpiece upon the canvas of football history!
  3. (0.391) A sensational diving catch at the boundary ropes, he has pulled that out of thin air, absolutely unbelievable athleticism!

QUERY: Vaibhav is magnificient at hitting boundaries at this age.
  1. (0.534) Young prodigy Vaibhav Sooryavanshi shows nerves of steel, smashing the ball past the fielder with incredible maturity!
  2. (0.361) Virat Kohli steps down the track and lofts it high into the night sky, that is absolutely massive, what a shot!
  3. (0.314) He started strength training to support his joints as he ages.

QUERY: Engineers deployed the new app update to all users.
  1. (0.924) Engineers deployed the

# Part 2 - OpenSearch

In [6]:
use_subproject("../opensearch_search")

from src.data import get_documents
from src.embedder import encode_texts
from src.client import create_collection, get_opensearch_client, delete_collection
from src.indexer import create_index_with_knn_mapping, index_documents
from src.search import keyword_search, semantic_search, hybrid_search
from utils.config import config

print("Collection name:", config.collection_name)
print("Index name:", config.index_name)
print("Region:", config.region_name)


Active sub-project: /Users/pateljay/Documents/GenAI/practical_07_vector_search_faiss_opensearch/opensearch_search
Collection name: practical7-vector-search
Index name: documents
Region: us-east-1


### Create the collection


In [8]:
# aws sts get-caller-identity -- Use this to find arn, userID and account
PRINCIPAL_ARN = "arn:aws:iam::378311506052:user/Master_05"

endpoint = create_collection(PRINCIPAL_ARN)
print("Collection endpoint:", endpoint)

client = get_opensearch_client(endpoint)

Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Waiting for collection to become ACTIVE...
Collection endpoint: https://yvo7el17izurhwj2er2b.us-east-1.aoss.amazonaws.com


### Create the index and index the 50 documents

In [9]:
client = get_opensearch_client(endpoint)

In [10]:
create_index_with_knn_mapping(client)

Created index 'documents'.


In [11]:
documents = get_documents()
print(documents[0])

{'text': 'Lionel Messi, the maestro of Rosario, writes his name into the heavens as the greatest player of all time!', 'category': 'Sports & Outdoors'}


In [12]:
documents = get_documents()

print(type(documents))
print(len(documents))
print(documents[0])

texts = [doc["text"] for doc in documents]

print(len(texts))
print(texts[0])

print(documents[0].keys())

<class 'list'>
50
{'text': 'Lionel Messi, the maestro of Rosario, writes his name into the heavens as the greatest player of all time!', 'category': 'Sports & Outdoors'}
50
Lionel Messi, the maestro of Rosario, writes his name into the heavens as the greatest player of all time!
dict_keys(['text', 'category'])


In [13]:
print(embeddings.shape)

(100, 384)


In [14]:
from utils.config import config
print(config.embedding_dimension)

384


In [15]:
try:
    index_documents(client, documents, embeddings)
except Exception as e:
    print(type(e))
    if hasattr(e, "errors"):
        print(e.errors[0])

Indexed 50 documents.


In [16]:
# Create the OpenSearch index (only once)
create_index_with_knn_mapping(client)

# Load the sample documents
documents = get_documents()

# Extract the text from each document
texts = [doc["text"] for doc in documents]

# Generate embeddings for every document
embeddings = encode_texts(texts)

# Store documents + vectors in OpenSearch
index_documents(client, documents, embeddings)

Index 'documents' already exists -- reusing it.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 50 documents.


### Compare keyword, semantic, and hybrid search

In [18]:
query = "Messi Ronaldo Mbappe Neymar Haaland legendary football players striking and electrifying the stadium"
query_vector = encode_texts([query])[0]

keyword_results = keyword_search(client, query, k=5)
semantic_results = semantic_search(client, query_vector, k=5)
hybrid_results = hybrid_search(client, query, query_vector, k=5)

print("===== Keyword Search =====")
for r in keyword_results:
    print(f"Score: {r['score']:.3f}")
    print(f"Category: {r['category']}")
    print(f"Text: {r['text']}")

print("\n===== Semantic Search =====")
for r in semantic_results:
    print(f"Score: {r['score']:.3f}")
    print(f"Category: {r['category']}")
    print(f"Text: {r['text']}")

print("\n===== Hybrid Search =====")
for r in hybrid_results:
    print(f"Score: {r['score']:.4f}")
    print(f"Category: {r['category']}")
    print(f"Text: {r['text']}")

===== Keyword Search =====

===== Semantic Search =====

===== Hybrid Search =====


## 3. FAISS vs. OpenSearch — what actually changed?

**FAISS** stores vectors in your application's memory and performs
  vector search locally. It is fast and ideal for experiments or
  small-scale applications.

- **OpenSearch** stores vectors in a managed cloud service. Instead of
  searching locally, your application sends a request to OpenSearch,
  which performs the search and returns the results.

Another major advantage of OpenSearch is that it supports **Hybrid
Search**, where vector search can be combined with traditional keyword
(BM25) search. FAISS only performs vector similarity search and cannot
search using keywords by itself.

**In short:**

- **FAISS** → Local vector search, simple, fast, great for development.
- **OpenSearch** → Managed vector search with additional features like
  keyword search, hybrid search, filtering, scalability, and production
  deployment.